# Entrenamiento YOLO
Usa el `dataset_p<N>_*.zip` que deja `python xd.py dataset N --drive` en Drive (`PoC-Labelling/datasets`).
En la celda 1 pon `PROYECTO` (y `DS` si no es el mas reciente). Al terminar, en el host:
`python xd.py registrar N --run ultimo --activar`.

`ultralytics` fijo en 8.4.174, el mismo del hub: un modelo de otra version puede no cargar en `ver`, `hijo` ni aceptacion.

In [ ]:
# === PARAMETROS (lo unico que se edita) ===
PROYECTO = 0        # numero del proyecto en Label Studio (obligatorio)
DS = ''             # '' = el dataset_p<PROYECTO>_*.zip mas reciente en Drive/datasets
FORMATO = 'det'     # 'det' o 'seg' (el zip debe traer esa carpeta)
MODELO = ''         # '' = yolo11n.pt (det) / yolo11n-seg.pt (seg)
IMGSZ, EPOCHS, PATIENCE, BATCH = 960, 100, 30, 16
CONF_CONTEO = 0.25  # umbral del KPI de conteo
assert PROYECTO > 0, 'pon PROYECTO = <numero del proyecto en Label Studio>'

!pip -q install ultralytics==8.4.174
from google.colab import drive
drive.mount('/content/drive')
import json, zipfile, yaml
from pathlib import Path

BASE = Path('/content/drive/MyDrive/PoC-Labelling')
if not DS:
    zs = sorted((BASE / 'datasets').glob(f'dataset_p{PROYECTO}_*.zip'))
    assert zs, f'No hay dataset_p{PROYECTO}_*.zip en {BASE / "datasets"}'
    DS = zs[-1].name
zp = BASE / 'datasets' / DS
d = Path('/content/ds') / zp.stem
with zipfile.ZipFile(zp) as z:
    z.extractall(d)
yml = d / FORMATO / 'data.yaml'
assert yml.exists(), f'{DS} no trae la carpeta {FORMATO}'
with open(yml) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = str(d / FORMATO)
with open(yml, 'w') as f:
    yaml.safe_dump(cfg, f)
RESUMEN = json.loads((d / 'resumen.json').read_text()) if (d / 'resumen.json').exists() else {}
MODELO = MODELO or ('yolo11n-seg.pt' if FORMATO == 'seg' else 'yolo11n.pt')
RUN = zp.stem if f'_{FORMATO}_' in zp.stem else f'{zp.stem}_{FORMATO}'
print(f'Dataset {DS} | run {RUN} | modelo {MODELO} | imagenes {RESUMEN.get("imagenes")} | '
      f'correcciones {RESUMEN.get("n_correcciones")}')

In [ ]:
from ultralytics import YOLO

_ = YOLO(MODELO).train(data=str(yml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, plots=True,
                       project=str(BASE / 'runs'), name=RUN, exist_ok=True)

In [ ]:
import csv, datetime, ultralytics
from collections import Counter
from IPython.display import Image, display
from ultralytics import YOLO

run = BASE / 'runs' / RUN
best = YOLO(run / 'weights' / 'best.pt')
names = best.names
m = best.val(data=str(yml), imgsz=IMGSZ, plots=False)
por_clase = {names[i]: {} for i in names}
for k, c in enumerate(m.box.ap_class_index):
    por_clase[names[int(c)]].update(ap50=round(float(m.box.ap50[k]), 4), ap50_95=round(float(m.box.ap[k]), 4))

# KPI: conteo por imagen en val, prediccion (CONF_CONTEO) vs anotacion
err, real_tot, exactas = Counter(), Counter(), 0
imgs = sorted((d / FORMATO / 'images' / 'val').glob('*'))
for im in imgs:
    real = Counter(int(l.split()[0]) for l in open(d / FORMATO / 'labels' / 'val' / (im.stem + '.txt')) if l.strip())
    pred = Counter(best.predict(str(im), imgsz=IMGSZ, conf=CONF_CONTEO, verbose=False)[0].boxes.cls.int().tolist())
    real_tot.update(real)
    dif = {c: pred[c] - real[c] for c in set(real) | set(pred)}
    exactas += all(v == 0 for v in dif.values())
    for c, v in dif.items():
        err[c] += abs(v)
for i, nm in names.items():
    por_clase[nm].update(error_abs=err[i], mae=round(err[i] / max(1, len(imgs)), 3), instancias_val=real_tot[i])

with open(run / 'results.csv') as f:
    epocas = sum(1 for _ in csv.DictReader(f))
met = {"esquema": 1, "creado": datetime.datetime.now().isoformat(timespec="seconds"),
       "proyecto": PROYECTO, "dataset": DS, "resumen_dataset": RESUMEN, "formato": FORMATO,
       "modelo_base": MODELO, "ultralytics": ultralytics.__version__,
       "params": {"imgsz": IMGSZ, "epochs": EPOCHS, "patience": PATIENCE, "batch": BATCH, "conf_conteo": CONF_CONTEO},
       "epocas_ejecutadas": epocas,
       "conteo": {"exactas": exactas, "imagenes": len(imgs), "tasa": round(exactas / max(1, len(imgs)), 3)},
       "map50": round(float(m.box.map50), 4), "map50_95": round(float(m.box.map), 4), "por_clase": por_clase}
if FORMATO == 'seg':
    met["seg"] = {"map50": round(float(m.seg.map50), 4), "map50_95": round(float(m.seg.map), 4)}
(run / 'weights' / 'metricas.json').write_text(json.dumps(met, ensure_ascii=True, indent=1))

print(f"\nCONTEO EXACTO {exactas}/{len(imgs)} ({met['conteo']['tasa']:.0%}) | mAP50 {met['map50']:.3f} | "
      f"mAP50-95 {met['map50_95']:.3f} | epocas {epocas}")
print(f"{'clase':<14}{'AP50':>7}{'err':>6}{'MAE':>7}{'inst':>6}")
for nm, v in por_clase.items():
    print(f"{nm:<14}{v.get('ap50', 0):>7.3f}{v['error_abs']:>6}{v['mae']:>7.3f}{v['instancias_val']:>6}")
print('metricas ->', run / 'weights' / 'metricas.json')
display(Image(str(run / 'results.png')))
display(Image(str(run / 'confusion_matrix.png')))